 pip install peft 
 pip install datasets 
 pip install trl 
 pip install transformers

In [28]:
import os
import torch
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, PeftModel  # PEFT freezes most weights; trains only selected layers (e.g., LoRA adapters)
from trl import SFTTrainer


In [29]:
# Model and dataset
model_name = "NousResearch/Llama-2-7b-chat-hf"
dataset_name = "mlabonne/guanaco-llama2-1k"
new_model = "Llama-2-7b-chat-finetune"

# LoRA/QLoRA Config
loran_r = 64
lora_alpha = 16
lora_dropout = 0.1
use_4bit = True
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quantization = False

# Training Arguments
output_dir = "./results/"
num_train_epochs = 1
fp16 = False
bf16 = False
per_device_train_batch_size = 4
per_device_eval_batch_size = 4
gradient_accumulation_steps = 1
gradient_checkpointing = True
max_grad_norm = 0.3
learning_rate = 2e-5
weight_decay = 0.001
optimizer = "paged_adamw_32bit"
scheduler = "cosine"
max_steps = -1
warmup_ratio = 0.03
group_by_length = True
save_steps = 0
logging_steps = 25

# SFT Trainer-specific
max_seq_length = None
packing = False

# Set device map
device_map = {"": 0}

# training model 

In [30]:
# Load dataset
dataset = load_dataset(dataset_name, split="train")
# Configure compute dtype
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    use_nested_quant=use_nested_quantization,
)

# if torch.cuda.is_available():
#     major, _ = torch.cuda.get_device_capability()
#     if major >= 8:
#         print("=" * 80)
#         print("GPU is compatible with bfloat16")
#         print("=" * 80)
# else:
#     print("CUDA not available — running on CPU or incompatible setup.")



# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map,
)
model.config.use_cache = False  # Disable cache for training
model.config.pretraining_tp = 1

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# LoRA config
peft_config = LoraConfig(
    r=loran_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    task_type="CAUSAL_LM",
    bias="none",
)

# TrainingArguments
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optimizer,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=scheduler,
    report_to="tensorboard",
)

# Train with SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=training_arguments,
    peft_config=peft_config,
    max_seq_length=max_seq_length,
    packing=packing,
    dataset_text_field="text",
)

trainer.train()
# trainer.model.save_pretrained(new_model)

ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`

In [31]:
# Ignore warnings
logging.set_verbosity(logging.CRITICAL)

# Run text generation pipeline with our next model
prompt = "What is a large language model?"
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

NameError: name 'model' is not defined